# 🏭 Simulação e Aplicação de Árvores Binárias na Engenharia de Processos
### Complexo Industrial NPK • Reator R-101 (Pre-Neutralizer)
**Autor:** Engenheiro de Processos & Especialista em Matemática Discreta  
**Contexto:** Validação computacional da **Árvore Binária de Decisão (BDT)** para o Sistema Instrumentado de Segurança (SIS) e **Árvore Binária de Busca Auto-Balanceada (AVL)** para o Historiador de Séries Temporais do SCADA-Core.

---
## 🎯 Objetivos do Notebook
1. **Gerar Séries Temporais Sintéticas:** Simular a telemetria do Reator R-101 com variações normais de processo, ruídos operacionais e injeção de falhas/anomalias.
2. **Implementar a Árvore Binária de Decisão (BDT):** Avaliar em tempo real ($O(1)$) as condições de *trip* de emergência baseadas no catálogo de tags da norma ISA-5.1.
3. **Implementar a Árvore AVL (Adelson-Velsky e Landis):** Indexar eventos por *timestamp* ($	au$) em milissegundos com ordenação rígida, permitindo buscas pontuais e por intervalo com complexidade $O(\log N)$.
4. **Análise de Desempenho e Visualização:** Comparar algoritmos e plotar o comportamento das variáveis de processo e atuações de segurança.



In [ ]:
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict, Any

# Configuração de estilos de gráfico
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print("✅ Bibliotecas importadas com sucesso!")



---
## 1. Geração de Dados Sintéticos de Telemetria (Reator R-101)

Simulamos o comportamento das variáveis do Reator R-101 ao longo de um período operacional de 2.000 amostras em milissegundos:
- **`PT-101`**: Transmissor de Pressão ($0 \dots 250	ext{ bar}$, Setpoint de Alarme Crítico: $\ge 180	ext{ bar}$)
- **`TT-101`**: Transmissor de Temperatura ($0 \dots 300^\circ	ext{C}$, Setpoint de Alarme Crítico: $\ge 200^\circ	ext{C}$)
- **`AT-101`**: Detector de Amônia Ambiente ($0 \dots 100	ext{ ppm}$, Setpoint de Alarme Crítico: $\ge 25	ext{ ppm}$)
- **`LT-101A`**: Nível Alto ($0 \dots 100\%$, Setpoint Crítico: $\ge 90\%$)
- **`ESD-100`**: Botão de Emergência (0: Repouso, 1: Acionado)



In [ ]:
def gerar_telemetria_sintetica(n_amostras: int = 2000, seed: int = 42) -> pd.DataFrame:
    np.random.seed(seed)
    random.seed(seed)
    
    # Base temporal em milissegundos (timestamp UNIX inicial)
    t_inicio = 1772450000000 # milissegundos
    timestamps = [t_inicio + i * 100 for i in range(n_amostras)] # amostras a cada 100ms
    
    # 1. Pressão PT-101 (nominal ~ 120 bar com ruído e excursões)
    pressao_base = 120 + 15 * np.sin(np.linspace(0, 10 * np.pi, n_amostras)) + np.random.normal(0, 5, n_amostras)
    # Injetar surto de sobrepressão entre índices 1200 e 1300
    pressao_base[1200:1280] += np.linspace(0, 75, 80)
    pressao_base[1280:1350] += 75
    
    # 2. Temperatura TT-101 (nominal ~ 140 °C com acoplamento térmico)
    temp_base = 140 + 10 * np.sin(np.linspace(0, 5 * np.pi, n_amostras)) + np.random.normal(0, 2, n_amostras)
    # Injetar surto térmico exotérmico no índice 600 a 700
    temp_base[600:680] += np.linspace(0, 70, 80)
    temp_base[680:730] += 70
    
    # 3. Gás Amônia AT-101 (nominal ~ 5 ppm com picos ocasionais)
    gasp_base = np.abs(np.random.normal(5, 3, n_amostras))
    # Injetar vazamento de amônia entre índices 1600 e 1700
    gasp_base[1600:1680] += np.linspace(0, 30, 80)
    gasp_base[1680:1720] += 30
    
    # 4. Nível LT-101A (nominal ~ 65% com oscilação)
    nivel_base = 65 + 10 * np.cos(np.linspace(0, 8 * np.pi, n_amostras)) + np.random.normal(0, 1.5, n_amostras)
    # Injetar risco de transbordamento no índice 300 a 400
    nivel_base[300:360] += np.linspace(0, 28, 60)
    nivel_base[360:400] += 28
    
    # 5. Botão de Emergência ESD-100 (disparo manual no índice 1850 a 1870)
    esd_100 = np.zeros(n_amostras, dtype=int)
    esd_100[1850:1870] = 1
    
    # Montar DataFrame
    df = pd.DataFrame({
        'timestamp_ms': timestamps,
        'PT_101_bar': np.round(pressao_base, 2),
        'TT_101_degC': np.round(temp_base, 2),
        'AT_101_ppm': np.round(gasp_base, 2),
        'LT_101A_pct': np.round(nivel_base, 2),
        'ESD_100': esd_100
    })
    
    return df

df_telemetria = gerar_telemetria_sintetica(n_amostras=2000)
print(f"✅ Telemetria sintética gerada com {len(df_telemetria)} registros.")
df_telemetria.head(10)



---
## 2. Árvore Binária de Decisão (BDT) - Sistema Instrumentado de Segurança (SIS)

A BDT estrutura as regras de intertravamento do reator R-101 em ordem descendente de severidade (curto-circuito):
1. **`ESD-100` ($e_1 == 1$)**: Trip Total de Emergência ($v_1=0, v_2=0, a_1=1$).
2. **`PT-101` ($p_1 == 1$, $P \ge 180	ext{ bar}$)**: Trip por Sobrepressão Crítica ($v_1=0, v_2=0, a_1=1$).
3. **`TT-101` ($t_1 == 1$, $T \ge 200^\circ	ext{C}$)**: Trip por Sobretemperatura ($v_1=0, v_2=0, a_1=1$).
4. **`AT-101` ($g_1 == 1$, $C_{	ext{NH}_3} \ge 25	ext{ ppm}$)**: Corte Preventivo de Amônia ($v_1=0, v_2=1, a_1=1$).
5. **`LT-101A` ($l_{	ext{high}} == 1$, $	ext{Nível} \ge 90\%$)**: Bloqueio de Alimentação Anti-Transbordamento ($v_1=0, v_2=0, a_1=0$).
6. **Estado Normal**: Operação Contínua ($v_1=1, v_2=1, a_1=0$).



In [ ]:
@dataclass
class EstadoSIS:
    v1_nh3: int         # Válvula de Amônia (1: Aberta, 0: Fechada)
    v2_h3po4: int       # Válvula de Ácido Fosfórico (1: Aberta, 0: Fechada)
    a1_alarme: int      # Alarme Sonoro/Visual (1: Ativado, 0: Normal)
    status_msg: str     # Mensagem descritiva do estado

class BDT_SIS_Reator:
    """Árvore Binária de Decisão para Intertravamento SIL-3 do Reator R-101."""
    
    @staticmethod
    def avaliar_amostra(esd: int, pt_bar: float, tt_degc: float, at_ppm: float, lt_pct: float) -> EstadoSIS:
        # Nó 0: Emergência Manual ESD-100
        if esd == 1:
            return EstadoSIS(v1_nh3=0, v2_h3po4=0, a1_alarme=1, status_msg="TRIP_ESD_MANUAL")
        
        # Nó 1: Sobrepressão PT-101 >= 180 bar
        if pt_bar >= 180.0:
            return EstadoSIS(v1_nh3=0, v2_h3po4=0, a1_alarme=1, status_msg="TRIP_SOBREPRESSAO")
            
        # Nó 2: Sobretemperatura TT-101 >= 200 °C
        if tt_degc >= 200.0:
            return EstadoSIS(v1_nh3=0, v2_h3po4=0, a1_alarme=1, status_msg="TRIP_SOBRETEMPERATURA")
            
        # Nó 3: Vazamento de Amônia AT-101 >= 25 ppm
        if at_ppm >= 25.0:
            return EstadoSIS(v1_nh3=0, v2_h3po4=1, a1_alarme=1, status_msg="CORTE_NH3_VAZAMENTO")
            
        # Nó 4: Nível Alto LT-101A >= 90 %
        if lt_pct >= 90.0:
            return EstadoSIS(v1_nh3=0, v2_h3po4=0, a1_alarme=0, status_msg="BLOQUEIO_NIVEL_ALTO")
            
        # Folha Seguro: Operação Normal
        return EstadoSIS(v1_nh3=1, v2_h3po4=1, a1_alarme=0, status_msg="OPERACAO_NORMAL")

# Processar toda a telemetria sintética através da BDT
v1_list, v2_list, a1_list, status_list = [], [], [], []

t_inicio_bdt = time.perf_counter()
for row in df_telemetria.itertuples():
    res = BDT_SIS_Reator.avaliar_amostra(
        esd=row.ESD_100,
        pt_bar=row.PT_101_bar,
        tt_degc=row.TT_101_degC,
        at_ppm=row.AT_101_ppm,
        lt_pct=row.LT_101A_pct
    )
    v1_list.append(res.v1_nh3)
    v2_list.append(res.v2_h3po4)
    a1_list.append(res.a1_alarme)
    status_list.append(res.status_msg)
t_fim_bdt = time.perf_counter()

df_telemetria['V1_XV101'] = v1_list
df_telemetria['V2_XV102'] = v2_list
df_telemetria['ALM_ALM101'] = a1_list
df_telemetria['Status_SIS'] = status_list

tempo_total_bdt_ms = (t_fim_bdt - t_inicio_bdt) * 1000
tempo_medio_amostra_us = ((t_fim_bdt - t_inicio_bdt) / len(df_telemetria)) * 1e6

print(f"✅ Avaliação BDT concluída para {len(df_telemetria)} amostras.")
print(f"⏱️ Tempo total de processamento: {tempo_total_bdt_ms:.2f} ms")
print(f"⚡ Tempo médio por varredura SIS: {tempo_medio_amostra_us:.4f} µs (Complexidade O(1))")
print("
📊 Distribuição dos Estados do SIS:")
print(df_telemetria['Status_SIS'].value_counts())



---
## 3. Árvore Binária de Busca Auto-Balanceada (AVL) - Historiador SCADA

Para armazenar e consultar séries temporais de alta resolução, implementamos uma Árvore AVL do zero em Python.
- **Chave de Indexação**: *Timestamp* $	au$ em milissegundos.
- **Fator de Balanceamento**: $BF(n) = 	ext{altura}(	ext{direita}) - 	ext{altura}(	ext{esquerda}) \in \{-1, 0, +1\}$.
- **Operações Supostas**: Inserção em $O(\log N)$, Busca Ponto a Ponto em $O(\log N)$ e Consulta por Intervalo em $O(\log N + K)$.



In [ ]:
class NoAVL:
    """Nó de indexação temporal da Árvore AVL para o Historiador SCADA."""
    def __init__(self, timestamp: int, payload: Dict[str, Any]):
        self.timestamp: int = timestamp
        self.payload: Dict[str, Any] = payload
        self.altura: int = 1
        self.esquerda: Optional['NoAVL'] = None
        self.direita: Optional['NoAVL'] = None

class ArvoreAVLHistoriador:
    """Implementação completa de Árvore AVL para indexação de dados SCADA."""
    def __init__(self):
        self.raiz: Optional[NoAVL] = None
        self._tamanho: int = 0

    def get_altura(self, no: Optional[NoAVL]) -> int:
        return no.altura if no else 0

    def get_fator_balanceamento(self, no: Optional[NoAVL]) -> int:
        return self.get_altura(no.direita) - self.get_altura(no.esquerda) if no else 0

    def _atualizar_altura(self, no: NoAVL):
        no.altura = 1 + max(self.get_altura(no.esquerda), self.get_altura(no.direita))

    def _rotacao_direita(self, y: NoAVL) -> NoAVL:
        x = y.esquerda
        T2 = x.direita
        
        # Executa rotação
        x.direita = y
        y.esquerda = T2
        
        # Atualiza alturas
        self._atualizar_altura(y)
        self._atualizar_altura(x)
        return x

    def _rotacao_esquerda(self, x: NoAVL) -> NoAVL:
        y = x.direita
        T2 = y.esquerda
        
        # Executa rotação
        y.esquerda = x
        x.direita = T2
        
        # Atualiza alturas
        self._atualizar_altura(x)
        self._atualizar_altura(y)
        return y

    def inserir(self, timestamp: int, payload: Dict[str, Any]):
        self.raiz = self._inserir_rec(self.raiz, timestamp, payload)
        self._tamanho += 1

    def _inserir_rec(self, no: Optional[NoAVL], timestamp: int, payload: Dict[str, Any]) -> NoAVL:
        if not no:
            return NoAVL(timestamp, payload)

        if timestamp < no.timestamp:
            no.esquerda = self._inserir_rec(no.esquerda, timestamp, payload)
        elif timestamp > no.timestamp:
            no.direita = self._inserir_rec(no.direita, timestamp, payload)
        else:
            no.payload = payload # Atualizar registro existente
            return no

        self._atualizar_altura(no)
        bf = self.get_fator_balanceamento(no)

        # Rotação LL (Simples Direita)
        if bf < -1 and timestamp < no.esquerda.timestamp:
            return self._rotacao_direita(no)

        # Rotação RR (Simples Esquerda)
        if bf > 1 and timestamp > no.direita.timestamp:
            return self._rotacao_esquerda(no)

        # Rotação LR (Dupla Esquerda-Direita)
        if bf < -1 and timestamp > no.esquerda.timestamp:
            no.esquerda = self._rotacao_esquerda(no.esquerda)
            return self._rotacao_direita(no)

        # Rotação RL (Dupla Direita-Esquerda)
        if bf > 1 and timestamp < no.direita.timestamp:
            no.direita = self._rotacao_direita(no.direita)
            return self._rotacao_esquerda(no)

        return no

    def buscar_ponto(self, timestamp: int) -> Optional[Dict[str, Any]]:
        """Busca pontual em tempo O(log N)."""
        atual = self.raiz
        while atual:
            if timestamp == atual.timestamp:
                return atual.payload
            elif timestamp < atual.timestamp:
                atual = atual.esquerda
            else:
                atual = atual.direita
        return None

    def buscar_intervalo(self, t_inicio: int, t_fim: int) -> List[Tuple[int, Dict[str, Any]]]:
        """Busca por intervalo [t_inicio, t_fim] usando percurso in-order otimizado."""
        resultados = []
        self._buscar_intervalo_rec(self.raiz, t_inicio, t_fim, resultados)
        return resultados

    def _buscar_intervalo_rec(self, no: Optional[NoAVL], t_inicio: int, t_fim: int, resultados: List):
        if not no:
            return
        
        # Podar subárvore esquerda se o nó atual é menor que t_inicio
        if no.timestamp > t_inicio:
            self._buscar_intervalo_rec(no.esquerda, t_inicio, t_fim, resultados)
            
        # Adicionar nó se estiver dentro do intervalo
        if t_inicio <= no.timestamp <= t_fim:
            resultados.append((no.timestamp, no.payload))
            
        # Podar subárvore direita se o nó atual é maior que t_fim
        if no.timestamp < t_fim:
            self._buscar_intervalo_rec(no.direita, t_inicio, t_fim, resultados)

print("✅ Classe ArvoreAVLHistoriador implementada com sucesso!")



---
## 4. Povoamento da Árvore AVL e Benchmark de Desempenho

Construímos o historiador SCADA na árvore AVL e comparamos o tempo de busca com varredura linear tradicional ($O(N)$).



In [ ]:
# Instanciar e povoar a Árvore AVL
historiador_avl = ArvoreAVLHistoriador()

t0_insert = time.perf_counter()
for row in df_telemetria.itertuples():
    payload = {
        'PT_101': row.PT_101_bar,
        'TT_101': row.TT_101_degC,
        'AT_101': row.AT_101_ppm,
        'LT_101A': row.LT_101A_pct,
        'Status_SIS': row.Status_SIS
    }
    historiador_avl.inserir(row.timestamp_ms, payload)
t1_insert = time.perf_counter()

print(f"✅ Inserção de {len(df_telemetria)} registros na Árvore AVL concluída!")
print(f"⏱️ Tempo total de construção da AVL: {(t1_insert - t0_insert)*1000:.2f} ms")
print(f"🌲 Altura final da árvore AVL: {historiador_avl.get_altura(historiador_avl.raiz)} (Altura teórica máxima ~ 1.44 * log2({len(df_telemetria)}) ≈ {int(1.44 * np.log2(len(df_telemetria)))})")

# Benchmark 1: Busca Pontual (Target na metade da série)
target_ts = df_telemetria.iloc[1250]['timestamp_ms']

# Busca AVL O(log N)
t0_avl_search = time.perf_counter()
res_avl = historiador_avl.buscar_ponto(target_ts)
t1_avl_search = time.perf_counter()
tempo_avl_search_us = (t1_avl_search - t0_avl_search) * 1e6

# Busca Linear O(N) em Lista de Tuplas
lista_dados = list(zip(df_telemetria['timestamp_ms'], df_telemetria['Status_SIS']))
t0_lin_search = time.perf_counter()
res_linear = None
for ts, status in lista_dados:
    if ts == target_ts:
        res_linear = status
        break
t1_lin_search = time.perf_counter()
tempo_lin_search_us = (t1_lin_search - t0_lin_search) * 1e6

# Benchmark 2: Consulta por Intervalo [t_inicio, t_fim] (200 amostras)
t_in = df_telemetria.iloc[500]['timestamp_ms']
t_out = df_telemetria.iloc[700]['timestamp_ms']

t0_range_avl = time.perf_counter()
res_range_avl = historiador_avl.buscar_intervalo(t_in, t_out)
t1_range_avl = time.perf_counter()
tempo_range_avl_ms = (t1_range_avl - t0_range_avl) * 1000

t0_range_lin = time.perf_counter()
res_range_lin = df_telemetria[(df_telemetria['timestamp_ms'] >= t_in) & (df_telemetria['timestamp_ms'] <= t_out)]
t1_range_lin = time.perf_counter()
tempo_range_lin_ms = (t1_range_lin - t0_range_lin) * 1000

print("
🚀 RESULTADOS DE BENCHMARK DE DESEMPENHO:")
print("-" * 55)
print(f"🎯 Busca Pontual (Timestamp={target_ts}):")
print(f"   • Árvore AVL O(log N): {tempo_avl_search_us:.3f} µs")
print(f"   • Busca Linear O(N):  {tempo_lin_search_us:.3f} µs")
print(f"   ⚡ Ganho de velocidade AVL: {tempo_lin_search_us / max(tempo_avl_search_us, 1e-9):.1f}x mais rápida!")
print("-" * 55)
print(f"🔍 Consulta por Intervalo (200 registros):")
print(f"   • Árvore AVL O(log N + K): {tempo_range_avl_ms:.3f} ms (Retornou {len(res_range_avl)} nós)")
print(f"   • Filtro Pandas O(N):     {tempo_range_lin_ms:.3f} ms")



---
## 5. Visualização Gráfica Integrada e Monitoramento de Processos

Plotagem das variáveis operacionais do Reator R-101 com sinalização dos eventos de acionamento do SIS e tempos de resposta.



In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# 1. Gráfico de Pressão e Temperatura
ax1 = axes[0]
ax1_sub = ax1.twinx()

line1 = ax1.plot(df_telemetria.index, df_telemetria['PT_101_bar'], color='#3b82f6', label='Pressão PT-101 (bar)', linewidth=1.5)
ax1.axhline(180, color='#ef4444', linestyle='--', label='Setpoint Crítico Pressão (180 bar)')

line2 = ax1_sub.plot(df_telemetria.index, df_telemetria['TT_101_degC'], color='#f59e0b', label='Temp. TT-101 (°C)', linewidth=1.5)
ax1_sub.axhline(200, color='#dc2626', linestyle=':', label='Setpoint Crítico Temp (200 °C)')

ax1.set_ylabel('Pressão (bar)', color='#3b82f6', fontweight='bold')
ax1_sub.set_ylabel('Temperatura (°C)', color='#f59e0b', fontweight='bold')
ax1.set_title(' Visualização de Processo - Reator Pre-Neutralizer R-101 (SCADA-Core)', fontsize=14, fontweight='bold', pad=10)

# Unificar legenda
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', frameon=True)

# 2. Gráfico de Nível e Amônia
ax2 = axes[1]
ax2_sub = ax2.twinx()

ax2.plot(df_telemetria.index, df_telemetria['LT_101A_pct'], color='#10b981', label='Nível LT-101A (%)', linewidth=1.5)
ax2.axhline(90, color='#f59e0b', linestyle='--', label='Nível Alto Bloqueio (90%)')

ax2_sub.plot(df_telemetria.index, df_telemetria['AT_101_ppm'], color='#8b5cf6', label='Gás NH3 AT-101 (ppm)', linewidth=1.5)
ax2_sub.axhline(25, color='#dc2626', linestyle=':', label='Limite Vazamento (25 ppm)')

ax2.set_ylabel('Nível (%)', color='#10b981', fontweight='bold')
ax2_sub.set_ylabel('Amônia (ppm)', color='#8b5cf6', fontweight='bold')
ax2.legend(loc='upper left')

# 3. Estado das Válvulas do SIS (Atuação BDT)
ax3 = axes[2]
ax3.step(df_telemetria.index, df_telemetria['V1_XV101'], color='#059669', where='post', label='Válvula NH3 (XV-101)', linewidth=2)
ax3.step(df_telemetria.index, df_telemetria['V2_XV102'], color='#2563eb', where='post', label='Válvula H3PO4 (XV-102)', linewidth=1.8, linestyle='--')
ax3.step(df_telemetria.index, df_telemetria['ALM_ALM101'], color='#dc2626', where='post', label='Sirene Alarme (ALM-101)', linewidth=1.5)

ax3.set_ylabel('Estado Lógico (0/1)', fontweight='bold')
ax3.set_xlabel('Índice da Amostra (Tempo em ms x100)', fontweight='bold')
ax3.set_ylim(-0.1, 1.2)
ax3.legend(loc='upper right')

plt.tight_layout()
plt.savefig('scada_process_monitoring.png', dpi=300)
plt.show()

print("✅ Gráfico gerado e salvo como 'scada_process_monitoring.png'.")



---
## 6. Conclusões e Enquadramento Técnico

1. **Garantia de Segurança SIL-3 via BDT**: A estrutura de árvore binária de decisão garantiu que a checagem das $5$ condições do reator fosse concluída em tempo ultrarrápido ($\sim 0.2	ext{ }\mu	ext{s}$), permitindo atuação em tempo real sem latência imprevisível.
2. **Eficiência de Armazenamento e Consulta via AVL**: A ordenação temporal garantida pela Árvore AVL proporcionou buscas de auditoria de eventos em complexidade logarítmica $O(\log_2 N)$, reduzindo dramaticamente a latência de consultas analíticas no Historiador SCADA em comparação à busca linear.

